# Advanced TabM Variants for Playground Series S6E9

This notebook is for the next phase after the fast TabM submission beat the tree blends.

It runs stronger TabM variants with `FAST=False` style settings by default:

- stronger/default TabM baseline using Yusuke-style digit and frequency features
- optional Chris-formula feature variant
- optional multiple seeds
- TabM-only probability and rank blends

Use a Colab GPU runtime. A100 is ideal.

## 1. Install Dependencies

In [1]:
import subprocess
import sys

import torch

with open("constraints.txt", "w", encoding="utf-8") as f:
    f.write(f"torch=={torch.__version__}\n")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-c", "constraints.txt", "pytabkit==1.7.3"],
    check=True,
)

print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

torch 2.11.0+cu128
cuda available: True
gpu: NVIDIA A100-SXM4-80GB


## 2. Settings

Recommended first advanced run:

- `SEEDS = [42]`
- `RUN_BASE_FEATURES = True`
- `RUN_CHRIS_FEATURES = True`
- `USE_PWL = False`

After that, try more seeds. If `num_emb_type="pwl"` works in the installed `pytabkit` version, set `USE_PWL=True` for a separate run.

In [2]:
import glob
import inspect
import os
import time
import warnings

import numpy as np
import pandas as pd
import torch
from pytabkit import TabM_D_Classifier
from scipy.stats import norm, rankdata
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore", category=FutureWarning, message=".*force_all_finite.*")

SEED = 42
SEEDS = [42]
N_SPLITS = 5
TARGET = "Will_Buy_EV"
ID = "id"

RUN_BASE_FEATURES = True
RUN_CHRIS_FEATURES = True
USE_PWL = False

TABM_K = 32
BATCH_SIZE = 256
LR = 2e-3
N_EPOCHS = 200
PATIENCE = 16

def pick_device() -> str:
    if not torch.cuda.is_available():
        return "cpu"
    try:
        (torch.ones(8, device="cuda") @ torch.ones(8, device="cuda")).item()
        return "cuda"
    except Exception as error:
        print("CUDA unusable, falling back to CPU:", error)
        return "cpu"

DEVICE = pick_device()
print("device:", DEVICE)
if DEVICE == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))
print("TabM_D_Classifier signature:")
print(inspect.signature(TabM_D_Classifier))

device: cuda
gpu: NVIDIA A100-SXM4-80GB
TabM_D_Classifier signature:
(device: Optional[str] = None, random_state: Union[int, numpy.random.mtrand.RandomState, NoneType] = None, n_cv: int = 1, n_refit: int = 0, n_repeats: int = 1, val_fraction: float = 0.2, n_threads: Optional[int] = None, tmp_folder: Union[str, pathlib._local.Path, NoneType] = None, verbosity: int = 0, arch_type: Optional[str] = None, tabm_k: Optional[int] = None, num_emb_type: Optional[str] = None, num_emb_n_bins: Optional[int] = None, batch_size: Optional[int] = None, lr: Optional[float] = None, weight_decay: Optional[float] = None, n_epochs: Optional[int] = None, patience: Optional[int] = None, d_embedding: Optional[int] = None, d_block: Optional[int] = None, n_blocks: Union[str, int, NoneType] = None, dropout: Optional[float] = None, compile_model: Optional[bool] = None, allow_amp: Optional[bool] = None, tfms: Optional[List[str]] = None, gradient_clipping_norm: Union[float, Literal['none'], NoneType] = None, calibra

## 3. Load Data

In [3]:
def find_file(name: str) -> str:
    for root in ["/content", "/content/drive", "."]:
        hits = glob.glob(f"{root}/**/{name}", recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(f"Could not find {name}. Upload it or mount Drive first.")

TRAIN_PATH = find_file("train.csv")
TEST_PATH = find_file("test.csv")
SAMPLE_PATH = find_file("sample_submission.csv")

print("train:", TRAIN_PATH)
print("test:", TEST_PATH)
print("sample:", SAMPLE_PATH)

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)
y = train[TARGET].map({"No": 0, "Yes": 1}).astype(int).to_numpy()

print("train shape:", train.shape)
print("test shape:", test.shape)
print("positive rate:", round(float(y.mean()), 6))
assert sample.columns.tolist() == [ID, TARGET]

train: /content/drive/MyDrive/playground-series-s6e9/train.csv
test: /content/drive/MyDrive/playground-series-s6e9/test.csv
sample: /content/drive/MyDrive/playground-series-s6e9/sample_submission.csv
train shape: (668665, 15)
test shape: (286571, 14)
positive rate: 0.174645


## 4. Feature Builders

In [4]:
NUMERIC = [
    "Age",
    "Annual_Income_USD",
    "Daily_Commute_km",
    "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Environmental_Concern_Level",
]
CATEGORICAL = ["Gender", "City_Type", "Current_Car_Type", "Range_Anxiety_Level"]
FLAGS = ["Home_Charging_Possible", "Subsidy_Available"]
DIGIT_SPECS = {
    "Age": (1.0, 2),
    "Annual_Income_USD": (1.0, 6),
    "Daily_Commute_km": (10.0, 4),
    "Number_of_Cars_Owned": (1.0, 1),
    "Charging_Stations_Near_Home": (1.0, 2),
    "Charging_Stations_Near_Work": (1.0, 2),
    "Environmental_Concern_Level": (1.0, 1),
}

def recipe_score(frame: pd.DataFrame) -> np.ndarray:
    return (
        1.2 * frame["Annual_Income_USD"] / 1e5
        + 0.6 * frame["Environmental_Concern_Level"]
        + 2.0 * (frame["Subsidy_Available"] == "Yes")
        - 1.0 * (frame["Range_Anxiety_Level"] == "Medium")
        - 3.0 * (frame["Range_Anxiety_Level"] == "High")
    ).to_numpy(dtype=float)

def recipe_probability(frame: pd.DataFrame) -> np.ndarray:
    return np.clip(norm.cdf(recipe_score(frame) - 5.5), 1e-6, 1 - 1e-6)

def add_base_features(frame: pd.DataFrame, income_counts: pd.Series) -> pd.DataFrame:
    out = pd.DataFrame(index=frame.index)
    for column in NUMERIC:
        out[column] = pd.to_numeric(frame[column], errors="coerce").astype(float)
    for column in CATEGORICAL:
        out[column] = frame[column].fillna("__MISSING__").astype(str)
    for column in FLAGS:
        out[column] = (
            frame[column]
            .astype(str)
            .str.strip()
            .str.lower()
            .map({"yes": 1, "true": 1, "1": 1, "no": 0, "false": 0, "0": 0})
            .astype(float)
        )
    out["income_freq"] = frame["Annual_Income_USD"].map(income_counts).fillna(0).astype(float)
    for column, (scale, width) in DIGIT_SPECS.items():
        scaled = (pd.to_numeric(frame[column], errors="coerce") * scale).round().astype("Int64")
        for position in range(width):
            digit = ((scaled.abs() // 10**position) % 10).astype("Int64")
            out[f"digit__{column}__{position}"] = digit.astype("string").fillna("__MISSING__").astype(str)
    return out

def add_chris_features(out: pd.DataFrame, frame: pd.DataFrame) -> pd.DataFrame:
    out = out.copy()
    home = (frame["Home_Charging_Possible"] == "Yes").astype(float)
    subsidy = (frame["Subsidy_Available"] == "Yes").astype(float)
    chargers_total = frame["Charging_Stations_Near_Home"] + frame["Charging_Stations_Near_Work"]
    worry_score = (
        frame["Daily_Commute_km"]
        - 5 * frame["Charging_Stations_Near_Home"]
        - 5 * frame["Charging_Stations_Near_Work"]
        - 150 * home
    )
    score = recipe_score(frame)
    prob = recipe_probability(frame)
    out["worry_score"] = worry_score.astype(float)
    out["chargers_total"] = chargers_total.astype(float)
    out["income_x_subsidy"] = frame["Annual_Income_USD"] / 1e5 * subsidy
    out["concern_x_subsidy"] = frame["Environmental_Concern_Level"] * subsidy
    out["recipe_score"] = score
    out["recipe_probability"] = prob
    out["recipe_distance"] = score - 5.5
    out["abs_recipe_distance"] = np.abs(score - 5.5)
    out["worry_low_distance"] = worry_score - (-25)
    out["worry_high_distance"] = worry_score - 75
    out["abs_worry_low_distance"] = np.abs(worry_score - (-25))
    out["abs_worry_high_distance"] = np.abs(worry_score - 75)
    return out

income_counts = pd.concat([train["Annual_Income_USD"], test["Annual_Income_USD"]]).value_counts()
X_base = add_base_features(train, income_counts)
X_test_base = add_base_features(test, income_counts)
X_chris = add_chris_features(X_base, train)
X_test_chris = add_chris_features(X_test_base, test)

print("base shape:", X_base.shape)
print("chris shape:", X_chris.shape)
print("base categorical:", sum(not pd.api.types.is_numeric_dtype(X_base[c]) for c in X_base.columns))
print("chris categorical:", sum(not pd.api.types.is_numeric_dtype(X_chris[c]) for c in X_chris.columns))

base shape: (668665, 32)
chris shape: (668665, 44)
base categorical: 22
chris categorical: 22


## 5. Training Helpers

In [5]:
def make_tabm(seed: int) -> TabM_D_Classifier:
    kwargs = dict(
        device=DEVICE,
        random_state=seed,
        n_threads=4,
        verbosity=0,
        val_metric_name="1-auc_ovr",
        tabm_k=TABM_K,
        batch_size=BATCH_SIZE,
        lr=LR,
        n_epochs=N_EPOCHS,
        patience=PATIENCE,
    )
    if USE_PWL:
        kwargs["num_emb_type"] = "pwl"
    return TabM_D_Classifier(**kwargs)

def run_tabm_cv(name: str, X: pd.DataFrame, X_test: pd.DataFrame, seed: int):
    cat_cols = [column for column in X.columns if not pd.api.types.is_numeric_dtype(X[column])]
    oof = np.zeros(len(X), dtype=float)
    test_pred = np.zeros(len(X_test), dtype=float)
    fold_assignments = np.full(len(X), -1, dtype=np.int8)
    fold_scores = []
    fold_times = []
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    print(f"\n=== {name} | seed {seed} ===")
    print("features:", X.shape[1], "categorical:", len(cat_cols))
    for fold, (fit_idx, val_idx) in enumerate(skf.split(X, y), start=1):
        start = time.time()
        model = make_tabm(seed + fold)
        model.fit(
            X.iloc[fit_idx],
            y[fit_idx],
            X_val=X.iloc[val_idx],
            y_val=y[val_idx],
            cat_col_names=cat_cols,
        )
        oof[val_idx] = model.predict_proba(X.iloc[val_idx])[:, 1]
        test_pred += model.predict_proba(X_test)[:, 1] / N_SPLITS
        fold_assignments[val_idx] = fold
        fold_auc = roc_auc_score(y[val_idx], oof[val_idx])
        seconds = time.time() - start
        fold_scores.append(fold_auc)
        fold_times.append(seconds)
        print(f"fold {fold}: AUC={fold_auc:.6f}, seconds={seconds:.0f}")
    overall = roc_auc_score(y, oof)
    print(f"{name} seed {seed} OOF AUC: {overall:.6f}")
    print("fold mean:", round(float(np.mean(fold_scores)), 6), "std:", round(float(np.std(fold_scores)), 6))
    print("total minutes:", round(float(np.sum(fold_times) / 60), 2))
    assert (fold_assignments > 0).all()
    assert np.isfinite(oof).all()
    assert np.isfinite(test_pred).all()
    return {
        "name": name,
        "seed": seed,
        "oof": oof,
        "test": test_pred,
        "fold": fold_assignments,
        "auc": overall,
    }

def minmax(values):
    values = np.asarray(values, dtype=float)
    lo, hi = values.min(), values.max()
    if hi == lo:
        return np.zeros_like(values)
    return (values - lo) / (hi - lo)

def rank01(values):
    return minmax(rankdata(values, method="average"))

## 6. Run TabM Variants

This is the expensive cell. On A100, start with one seed. Add more seeds after you see the first result.

In [6]:
results = []

for seed in SEEDS:
    if RUN_BASE_FEATURES:
        results.append(run_tabm_cv("tabm_base_default", X_base, X_test_base, seed))
    if RUN_CHRIS_FEATURES:
        results.append(run_tabm_cv("tabm_chris_default", X_chris, X_test_chris, seed))

summary = pd.DataFrame(
    {
        "name": [r["name"] for r in results],
        "seed": [r["seed"] for r in results],
        "auc": [r["auc"] for r in results],
    }
).sort_values("auc", ascending=False)
summary


=== tabm_base_default | seed 42 ===
features: 32 categorical: 22
fold 1: AUC=0.943594, seconds=231
fold 2: AUC=0.944364, seconds=249
fold 3: AUC=0.945437, seconds=248
fold 4: AUC=0.944945, seconds=230
fold 5: AUC=0.944765, seconds=255
tabm_base_default seed 42 OOF AUC: 0.944596
fold mean: 0.944621 std: 0.000619
total minutes: 20.22

=== tabm_chris_default | seed 42 ===
features: 44 categorical: 22
fold 1: AUC=0.943380, seconds=227
fold 2: AUC=0.944584, seconds=253
fold 3: AUC=0.945589, seconds=237
fold 4: AUC=0.944874, seconds=228
fold 5: AUC=0.944756, seconds=253
tabm_chris_default seed 42 OOF AUC: 0.944609
fold mean: 0.944637 std: 0.000716
total minutes: 19.98


,name,seed,auc
1,tabm_chris_default,42,0.944609
0,tabm_base_default,42,0.944596


## 7. Save Individual Outputs

In [7]:
for result in results:
    suffix = f"{result['name']}_seed{result['seed']}"
    sub = pd.DataFrame({ID: test[ID].to_numpy(), TARGET: result["test"]})
    oof_frame = pd.DataFrame(
        {
            ID: train[ID].to_numpy(),
            "target": y,
            "fold": result["fold"],
            "prediction": result["oof"],
        }
    )
    sub.to_csv(f"submission_{suffix}.csv", index=False)
    oof_frame.to_csv(f"oof_{suffix}.csv", index=False)
    print(f"submission_{suffix}.csv", sub.shape, sub[TARGET].between(0, 1).all())
    print(f"oof_{suffix}.csv", oof_frame.shape)

submission_tabm_base_default_seed42.csv (286571, 2) True
oof_tabm_base_default_seed42.csv (668665, 4)
submission_tabm_chris_default_seed42.csv (286571, 2) True
oof_tabm_chris_default_seed42.csv (668665, 4)


## 8. TabM-Only Blends

These are honest OOF blends across TabM variants/seeds. Submit the best individual model first, then the best TabM-only blend.

In [8]:
if len(results) == 1:
    print("Only one result available, skipping blend creation.")
else:
    oof_matrix = pd.DataFrame({f"{r['name']}_seed{r['seed']}": r["oof"] for r in results})
    test_matrix = pd.DataFrame({f"{r['name']}_seed{r['seed']}": r["test"] for r in results})
    print("Correlation matrix:")
    print(oof_matrix.corr().round(6).to_string())

    avg_oof = oof_matrix.mean(axis=1).to_numpy()
    avg_test = test_matrix.mean(axis=1).to_numpy()
    rank_oof = pd.DataFrame({c: rank01(oof_matrix[c]) for c in oof_matrix.columns}).mean(axis=1).to_numpy()
    rank_test = pd.DataFrame({c: rank01(test_matrix[c]) for c in test_matrix.columns}).mean(axis=1).to_numpy()

    blend_scores = {
        "tabm_probability_average": roc_auc_score(y, avg_oof),
        "tabm_rank_average": roc_auc_score(y, rank_oof),
    }
    print("Blend scores:")
    for name, score in blend_scores.items():
        print(name, round(score, 6))

    pd.DataFrame({ID: test[ID], TARGET: avg_test}).to_csv("submission_tabm_probability_average.csv", index=False)
    pd.DataFrame({ID: test[ID], TARGET: rank_test}).to_csv("submission_tabm_rank_average.csv", index=False)
    print("Saved submission_tabm_probability_average.csv")
    print("Saved submission_tabm_rank_average.csv")

Correlation matrix:
                           tabm_base_default_seed42  tabm_chris_default_seed42
tabm_base_default_seed42                   1.000000                   0.997667
tabm_chris_default_seed42                  0.997667                   1.000000
Blend scores:
tabm_probability_average 0.944749
tabm_rank_average 0.94475
Saved submission_tabm_probability_average.csv
Saved submission_tabm_rank_average.csv


## 9. Save Everything to Drive

Run this if Drive is mounted. It copies the generated submissions and OOF files into `MyDrive/playground-series-s6e9/tabm_advanced_outputs`.

In [9]:
import shutil

drive_dir = "/content/drive/MyDrive/playground-series-s6e9/tabm_advanced_outputs"
if os.path.exists("/content/drive/MyDrive"):
    os.makedirs(drive_dir, exist_ok=True)
    for path in glob.glob("submission_tabm_*.csv") + glob.glob("oof_tabm_*.csv"):
        dest = os.path.join(drive_dir, os.path.basename(path))
        shutil.copy(path, dest)
        print(dest)
else:
    print("Drive is not mounted. Use from google.colab import drive; drive.mount('/content/drive') first.")

/content/drive/MyDrive/playground-series-s6e9/tabm_advanced_outputs/submission_tabm_base_default_seed42.csv
/content/drive/MyDrive/playground-series-s6e9/tabm_advanced_outputs/submission_tabm_rank_average.csv
/content/drive/MyDrive/playground-series-s6e9/tabm_advanced_outputs/submission_tabm_probability_average.csv
/content/drive/MyDrive/playground-series-s6e9/tabm_advanced_outputs/submission_tabm_chris_default_seed42.csv
/content/drive/MyDrive/playground-series-s6e9/tabm_advanced_outputs/submission_tabm_colab.csv
/content/drive/MyDrive/playground-series-s6e9/tabm_advanced_outputs/oof_tabm_base_default_seed42.csv
/content/drive/MyDrive/playground-series-s6e9/tabm_advanced_outputs/oof_tabm_colab.csv
/content/drive/MyDrive/playground-series-s6e9/tabm_advanced_outputs/oof_tabm_chris_default_seed42.csv
